<a href="https://colab.research.google.com/github/SunSpot-Tech/Flyrank_Internship/blob/main/work/notebooks/%20%20w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SunSpot-Tech/Flyrank_Internship/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import duckdb, os, pandas as pd, numpy as np
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"""
CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{os.environ["HF_TOKEN"]}');
""")

FACT = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet"
DIM  = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

feature_frame = con.sql(f"""
    SELECT
        f.content_hash_id, f.client_hash_id,
        DATE '2026-03-16' - d.content_updated_date AS days_since_last_update,
        SUM(CASE WHEN f.report_date < DATE '2026-03-16' THEN f.gsc_impressions ELSE 0 END) AS impressions_first_half,
        SUM(CASE WHEN f.report_date < DATE '2026-03-16' THEN f.gsc_clicks ELSE 0 END) AS clicks_first_half,
        AVG(CASE WHEN f.report_date < DATE '2026-03-16' THEN f.gsc_avg_position END) AS avg_position_first_half,
        SUM(CASE WHEN f.report_date >= DATE '2026-03-16' THEN f.gsc_clicks ELSE 0 END) AS clicks_second_half
    FROM read_parquet('{FACT}', hive_partitioning=1) f
    JOIN read_parquet('{DIM}') d ON f.content_hash_id = d.content_hash_id
    WHERE f.month = '2026-03' AND f.gsc_data_available IS TRUE
    GROUP BY f.content_hash_id, f.client_hash_id, d.content_updated_date
""").df()

# Fix: content_updated_date is a snapshot, not point-in-time — negative values are missing
feature_frame['days_since_last_update'] = feature_frame['days_since_last_update'].where(
    feature_frame['days_since_last_update'] >= 0, np.nan
)
feature_frame['is_declining'] = (feature_frame.clicks_second_half < feature_frame.clicks_first_half).astype(int)
feature_frame['ctr_first_half'] = feature_frame.clicks_first_half / feature_frame.impressions_first_half.replace(0, np.nan)

pos_bins = [0, 3, 6, 10, 20, 1000]
pos_labels = ['1-3', '4-6', '7-10', '11-20', '20+']
feature_frame['position_tier'] = pd.cut(feature_frame.avg_position_first_half, bins=pos_bins, labels=pos_labels)
tier_avg = feature_frame.groupby('position_tier', observed=True)['ctr_first_half'].mean()
feature_frame['tier_avg_ctr'] = feature_frame['position_tier'].map(tier_avg).astype(float)
feature_frame['ctr_gap'] = feature_frame['ctr_first_half'] < feature_frame['tier_avg_ctr']

features = ['impressions_first_half', 'clicks_first_half', 'avg_position_first_half',
            'ctr_first_half', 'days_since_last_update']

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

def assign_reason(row):
    is_stale = pd.notna(row['days_since_last_update']) and row['days_since_last_update'] >= 180
    is_visible = row['impressions_first_half'] >= 500
    has_ctr_gap = row['ctr_gap']
    if is_stale and is_visible and has_ctr_gap:
        return 2
    elif is_stale and is_visible:
        return 1
    elif has_ctr_gap:
        return 1
    else:
        return 0

print(feature_frame.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(176738, 12)


In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

model_df = feature_frame.dropna(subset=['avg_position_first_half']).copy()
X = model_df[features].fillna(-1)
y = model_df['is_declining']
groups = model_df['client_hash_id']
base_rate = y.mean()

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
test_df = model_df.iloc[test_idx].copy()

rf = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

print(f"Base rate: {base_rate:.3f}")
for k in (20, 50, 100):
    print(f"Precision@{k}: {precision_at_k(rf_scores, y_test, k):.3f}")

Base rate: 0.191
Precision@20: 1.000
Precision@50: 0.920
Precision@100: 0.880


In [ ]:
test_df['rf_score'] = rf_scores
test_df['reason_code'] = test_df.apply(assign_reason, axis=1)
reason_labels = {0: 'NONE', 1: 'STALE_OR_CTR_GAP', 2: 'STALE_AND_CTR_GAP'}
test_df['reason_code_label'] = test_df['reason_code'].map(reason_labels)

final_queue = test_df[test_df['reason_code'] > 0].sort_values('rf_score', ascending=False).copy()
final_queue['action'] = 'review_for_refresh'

print(f"Queue size: {len(final_queue)} rows")
print(final_queue['reason_code_label'].value_counts())
final_queue.head(10)

Queue size: 35220 rows
reason_code_label
STALE_OR_CTR_GAP     35217
STALE_AND_CTR_GAP        3
Name: count, dtype: int64


,content_hash_id,client_hash_id,days_since_last_update,impressions_first_half,clicks_first_half,avg_position_first_half,clicks_second_half,is_declining,ctr_first_half,position_tier,tier_avg_ctr,ctr_gap,rf_score,reason_code,reason_code_label,action
131984,content_9dd1a41ec2c2f835,client_73cda7b4e4f265ea,19.0,120.0,1.0,2.345866,0.0,1,0.008333,1-3,0.009315,True,0.894786,1,STALE_OR_CTR_GAP,review_for_refresh
132245,content_a2f2fbfd25382bfc,client_73cda7b4e4f265ea,19.0,120.0,1.0,1.838903,0.0,1,0.008333,1-3,0.009315,True,0.894786,1,STALE_OR_CTR_GAP,review_for_refresh
132394,content_a5a7228e5d08d494,client_73cda7b4e4f265ea,19.0,108.0,1.0,1.199815,0.0,1,0.009259,1-3,0.009315,True,0.894537,1,STALE_OR_CTR_GAP,review_for_refresh
161690,content_d7d6136d90e1eeab,client_73cda7b4e4f265ea,19.0,124.0,1.0,1.207107,0.0,1,0.008065,1-3,0.009315,True,0.890186,1,STALE_OR_CTR_GAP,review_for_refresh
74580,content_f46da42a202be839,client_73cda7b4e4f265ea,19.0,142.0,1.0,1.853281,1.0,0,0.007042,1-3,0.009315,True,0.890185,1,STALE_OR_CTR_GAP,review_for_refresh
132699,content_abdabdcef47993ef,client_73cda7b4e4f265ea,19.0,218.0,2.0,2.991364,1.0,1,0.009174,1-3,0.009315,True,0.889295,1,STALE_OR_CTR_GAP,review_for_refresh
108907,content_1abac4c74eb5df2b,client_73cda7b4e4f265ea,19.0,147.0,1.0,2.842522,0.0,1,0.006803,1-3,0.009315,True,0.889074,1,STALE_OR_CTR_GAP,review_for_refresh
107543,content_02480e89910fca47,client_73cda7b4e4f265ea,19.0,242.0,2.0,2.453886,0.0,1,0.008264,1-3,0.009315,True,0.888735,1,STALE_OR_CTR_GAP,review_for_refresh
43754,content_a249a1142207f604,client_73cda7b4e4f265ea,19.0,144.0,1.0,2.350461,0.0,1,0.006944,1-3,0.009315,True,0.888264,1,STALE_OR_CTR_GAP,review_for_refresh
92516,content_2acba08b6521d539,client_73cda7b4e4f265ea,19.0,248.0,2.0,1.859298,0.0,1,0.008065,1-3,0.009315,True,0.888053,1,STALE_OR_CTR_GAP,review_for_refresh


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The queue ranks pages by the Random Forest's predicted decline probability
(`rf_score`), restricted to pages that also carry an interpretable reason
code — so every ranked row has both a model score AND a plain-language
reason a human can check.

**Reason codes:**
- `STALE_OR_CTR_GAP` — either meaningfully stale with real visibility, or
  showing a CTR gap versus its position-tier peers
- `STALE_AND_CTR_GAP` — both conditions true (higher-confidence flag)

**Action:** `review_for_refresh` — a human decides whether to refresh,
deprioritize, or leave the page.

**Claim-ladder check:** these are measured, ranked outputs of a validated
model (grouped-split Precision@100 = 0.88), so "the model ranks/flags
pages at Precision@100 of 0.88" is the correct register — not "the model
finds pages that will decline."

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who uses this:** a content strategist or SEO analyst triaging which
pages to review first, out of a larger inventory than they have time to
check manually.

**What it's for:** decision support for prioritization — not a verdict
on any individual page, and not a claim about *why* a page will decline.

**Where it stops being valid:**
- **Scope:** trained/validated on March 2026 data for the clients present
  in this slice only. Not validated on other months or absent clients.
- **Known floor-effect distortion:** pages with `clicks_first_half = 0`
  mechanically guarantee `is_declining = 0` by the label's own
  arithmetic, inflating apparent precision on very low-volume pages.
- **Client-grouped validation gap:** grouped-split Precision@100 (0.88)
  is meaningfully below the naive-split figure (0.93) — 0.88 is the
  number to trust.
- **No causal design:** observational only. "Flagged" is not "refreshing
  will improve it" — no controlled experiment supports that stronger claim.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on any flagged row, a human should check:**
- Is `clicks_first_half` reasonably above zero (e.g. ≥5)? Below that, the
  flag may be a floor-effect artifact, not genuine risk.
- Does the page's position tier have enough comparable pages for the
  CTR-gap comparison to be meaningful?
- Is there a known, non-content reason for a dip (seasonality, a site
  migration) the model can't see?

**No-go list — never automate:**
- Auto-publishing or auto-editing content from the score alone.
- Auto-deprioritizing/removing a page without human review of the
  specific reason code and numbers.
- Communicating the score to a client as a performance guarantee.
- Extending this queue's client-level patterns to clients not present in
  the March 2026 training slice.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Signals these recommendations have gone stale:**
- A new month's base rate (currently 19.1%) shifts substantially from
  what the model was tuned to.
- Precision@K on a fresh month, measured the same way, drops meaningfully
  below the 0.88 (K=100) benchmark.
- A new client is added whose behavior wasn't represented in the training
  clients — the grouped-split design specifically doesn't guarantee this
  generalizes.
- The `content_updated_date` snapshot issue recurs in a future pull, re-verify the days-since-update fix before trusting staleness codes.

**Retrain trigger:** re-validate monthly using the same grouped-split
methodology, rather than assuming March 2026 patterns hold indefinitely.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
import json

os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

final_queue[['content_hash_id', 'client_hash_id', 'rf_score', 'reason_code_label',
             'action', 'impressions_first_half', 'clicks_first_half',
             'avg_position_first_half', 'days_since_last_update']].to_csv(
    'work/outputs/action_playbook_queue.csv', index=False
)

metrics = {
    'base_rate': round(float(base_rate), 3),
    'precision_at_20_grouped': round(float(precision_at_k(rf_scores, y_test, 20)), 3),
    'precision_at_50_grouped': round(float(precision_at_k(rf_scores, y_test, 50)), 3),
    'precision_at_100_grouped': round(float(precision_at_k(rf_scores, y_test, 100)), 3),
    'precision_at_100_naive': 0.93,
    'known_limitations': [
        'floor_effect_low_volume_pages',
        'client_grouped_gap_at_k100',
        'no_causal_design'
    ]
}
with open('work/outputs/model_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("Exported:", os.listdir('work/outputs'))

Exported: ['action_playbook_queue.csv', 'model_metrics.json']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.